# Lending Club -- cleaning & feature prep (rebuild v3)

**What this notebook does:** takes the interim DuckDB tables (`windowed`)
and produces the final modeling-ready table for Phase 1. This is the second
rebuild of the original cleaning notebook -- v2 incorporated all 14
`notebooks/02_eda/` categories; this version (v3) closes the last three gaps
a depth/sufficiency audit of that full EDA suite found: outlier treatment
was counted but never turned into a decision, row-level duplicate integrity
was never checked as part of *this* notebook's own validation, and the
categorical-encoding deferral was never written down. Every change is called
out explicitly, with the notebook and cell that justifies it.

This is still the **only** notebook in this repo that changes the data --
every `02_eda/` notebook stays read-only. Nothing here overwrites
`data/02_interim/`; this notebook only ever writes `data/03_processed/`.

**What changed vs. v2, and why:**

| Change | Reason | Source |
|---|---|---|
| 8 numeric fields winsorized at the 1st/99th percentile (`loan_amnt`, `dti`, `open_acc`, `revol_bal`, `tot_cur_bal`, `bc_open_to_buy`, `acc_open_past_24mths`, `num_actv_rev_tl`) | outlier counts existed (notebook 02 cells 3/4) but were never turned into a treatment decision -- `02_data_quality_integrity.ipynb` cell 8 makes that decision per-field; this cell applies it | notebook 02, cell 8 (gap-closure addendum) |
| Validation now also checks `id` uniqueness on the final table | v2's validation never independently re-checked row-level integrity on its own output | notebook 02, cell 6 (gap-closure addendum) |
| Categorical fields documented as deliberately unencoded, deferred to Phase 1 | this was always the intent but was never written down anywhere | notebook 14, cell 7 (gap-closure addendum) |

**What changed vs. the original (pre-v2) cleaning pass** -- unchanged from
v2, restated here for completeness:

| Change | Reason | Source |
|---|---|---|
| Dropped `avg_cur_bal` (raw + log) entirely | elevated VIF vs. `tot_cur_bal` -- redundant | notebook 09 |
| Log transform kept for only 9 of the original 16 fields | the other 7 didn't improve target correlation | notebook 09 |
| Added `issue_year` as a feature | `int_rate` PSI=0.140 across the window -- year carries real signal | notebook 10 |
| Added `addr_state_grouped` (small states bucketed to `OTHER`) | smallest states have single-digit loan counts | notebook 11 |
| `emp_title` confirmed excluded (was already out) | 317,489 distinct values, unusable without taxonomy work | notebook 12 |
| `grade`/`int_rate` kept, now documented as near-definitional | not independent causal drivers -- still valid predictors | notebook 13 |

**Cell index:**

| # | What it does | What to expect |
|---|---|---|
| 1 | Connect; define the updated retained-column scope | column counts, in/out lists |
| 2 | Median-impute the numeric fields with real (not structural) missingness | per-field impute counts |
| 3 | Mode-impute `emp_length`, add a missingness flag | impute count and flag rate |
| 4 | Log-transform only the 9 fields that showed real benefit (notebook 09) | before/after skew for those 9 only |
| 5 | Engineer `issue_year` and `addr_state_grouped` | new feature previews |
| 6 | Winsorize the 8 flagged fields at the 1st/99th percentile (notebook 02 cell 8's decision, applied for real) | bounds and rows affected per field |
| 7 | Assemble the final table, write to `data/03_processed/` | final shape |
| 8 | Validate the output, including a duplicate-id check and the encoding-deferral note | pass/fail checks |

**Final output of this notebook:** `data/03_processed/lendingclub_model_ready.parquet`
-- one row per windowed loan, every retained feature cleaned/imputed/
transformed, ready for Phase 1. Categorical fields are intentionally left
unencoded (see cell 8) -- encoding is a Phase 1 modeling decision, not a
Phase 0 one.

## Cell 1 -- connect and define the updated scope

**What / why:** stating the retained column list explicitly, with the
avg_cur_bal removal and the two new engineered features already reflected,
means this cell doubles as documentation of every column decision -- no
column enters or leaves the final table without being named here first.

**How:** connect read-only to the interim DuckDB file; define
`NUMERIC_KEEP` (down to 19 from 20 -- `avg_cur_bal` dropped), the 9-field
`LOG_FIELDS` subset (down from 16), the unchanged categorical set, and the
8-field `CAP_FIELDS` set this v3 rebuild adds (notebook 02 cell 8's
winsorizing decision).

**Expect:** confirmation counts matching this cell's stated lists.

In [2]:
import os, duckdb, pandas as pd, numpy as np
ASSETS_TABLES = "../../data/04_assets/tables"
ASSETS_PLOTS = "../../data/04_assets/plots"
os.makedirs(ASSETS_TABLES, exist_ok=True)
os.makedirs(ASSETS_PLOTS, exist_ok=True)
con = duckdb.connect("../../data/02_interim/lendingclub.duckdb", read_only=True)

NUMERIC_KEEP = ["loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_low",
                 "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec",
                 "revol_bal", "revol_util", "total_acc", "mort_acc",
                 "pub_rec_bankruptcies", "tot_cur_bal",
                 "bc_open_to_buy", "acc_open_past_24mths",
                 "mo_sin_old_rev_tl_op", "num_actv_rev_tl"]  # avg_cur_bal dropped (notebook 09, VIF)

LOG_FIELDS = ["annual_inc", "fico_range_low", "delinq_2yrs", "inq_last_6mths",
              "mo_sin_old_rev_tl_op", "mort_acc", "pub_rec", "pub_rec_bankruptcies",
              "total_acc"]  # only the 9 that improved target correlation (notebook 09)

CAP_FIELDS = ["loan_amnt", "dti", "open_acc", "revol_bal", "tot_cur_bal",
              "bc_open_to_buy", "acc_open_past_24mths", "num_actv_rev_tl"]
# the 8 fields notebook 02 cell 8 flagged "cap at 1st/99th percentile" --
# unbounded dollar/count fields where log didn't help (notebook 09) and no
# natural bound exists (unlike int_rate/revol_util/fico_range_low).

CATEGORICAL_KEEP = ["term", "grade", "emp_length", "home_ownership",
                     "verification_status", "purpose", "addr_state"]

n_rows = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
print(f"rows in scope: {n_rows:,}")
print(f"numeric columns retained: {len(NUMERIC_KEEP)} (was 20, avg_cur_bal dropped)")
print(f"of those, log-transformed: {len(LOG_FIELDS)} (was 16)")
print(f"of those, winsorized at 1st/99th percentile: {len(CAP_FIELDS)} (new in v3)")
print(f"categorical columns retained: {len(CATEGORICAL_KEEP)} (unchanged, left unencoded -- see cell 8)")


rows in scope: 1,195,879
numeric columns retained: 19 (was 20, avg_cur_bal dropped)
of those, log-transformed: 9 (was 16)
of those, winsorized at 1st/99th percentile: 8 (new in v3)
categorical columns retained: 7 (unchanged, left unencoded -- see cell 8)


**What the output shows:**
```
rows in scope: 1,195,879
numeric columns retained: 19 (was 20, avg_cur_bal dropped)
of those, log-transformed: 9 (was 16)
of those, winsorized at 1st/99th percentile: 8 (new in v3)
categorical columns retained: 7 (unchanged, left unencoded -- see cell 8)
```
1,195,879 rows in scope, matching the windowed population size from
every EDA notebook. The retained-column counts confirm both the notebook 09
structural changes and the new notebook 02 cell 8 capping decision are
applied at the scope-definition stage, before any imputation or transform
logic runs.

**Next:** imputing the numeric fields that actually have missing values --
the same set identified in notebook 02, minus `avg_cur_bal` since it's no
longer in scope.

## Cell 2 -- median-impute numeric fields with real missingness

**What / why:** notebook 02's data-quality profiling found 5 numeric fields
with real (non-structural) missingness, and ran an informativeness test on
each -- none showed a reliable missing-vs-populated bad-rate gap large enough
to justify anything beyond simple median imputation (unlike `emp_length`,
handled separately in cell 3). With `avg_cur_bal` now dropped from scope,
this rebuild imputes the remaining 4.

**How:** compute the median for each field from `windowed`, impute nulls
with it.

**Expect:** the same per-field impute counts as the v2 pass for
`bc_open_to_buy`, `revol_util`, `dti`, and `inq_last_6mths` -- this step
itself didn't change in v3, only cell 6 (winsorizing) is new.

In [3]:
MISSING_NUMERIC = ["bc_open_to_buy", "revol_util", "dti", "inq_last_6mths"]

medians = {}
for col in MISSING_NUMERIC:
    med, n_missing = con.sql(f"""
        SELECT median(TRY_CAST({col} AS DOUBLE)), sum(CASE WHEN TRY_CAST({col} AS DOUBLE) IS NULL THEN 1 ELSE 0 END)
        FROM windowed
    """).fetchone()
    medians[col] = med
    print(f"{col}: median={med:.2f}, rows to impute={n_missing:,}")


bc_open_to_buy: median=4612.00, rows to impute=12,383
revol_util: median=52.50, rows to impute=686
dti: median=17.87, rows to impute=223
inq_last_6mths: median=0.00, rows to impute=1


**What the output shows:**
```
bc_open_to_buy: median=4612.00, rows to impute=12,383
revol_util: median=52.50, rows to impute=686
dti: median=17.87, rows to impute=223
inq_last_6mths: median=0.00, rows to impute=1
```
Consistent with v2 and notebook 02's findings -- these are all
low-missingness, structurally unremarkable fields, safe for plain median
imputation. `avg_cur_bal`'s entry stays correctly absent.

**Next:** `emp_length` gets different treatment -- notebook 02 found its
missingness is genuinely informative (higher bad rate when missing), so it
gets mode imputation plus an explicit missingness flag rather than a silent
median fill.

## Cell 3 -- emp_length: mode imputation plus a missingness flag

**What / why:** unchanged from v2 -- notebook 02's informativeness test
(with its n>=1,000 reliability filter) found `emp_length`'s ~5.9%
missingness carries a real bad-rate signal, so a silent median-style fill
would destroy information a model could use. Mode imputation for the value
itself, plus a binary flag preserving "was this missing," keeps both pieces
of information.

**How:** compute the mode of `emp_length`, impute nulls with it, add
`emp_length_was_missing`.

**Expect:** the same numbers as v2 -- this step wasn't touched by the
gap-closure work.

In [4]:
emp_length_mode = con.sql("""
    SELECT emp_length, count(*) n FROM windowed WHERE emp_length IS NOT NULL
    GROUP BY 1 ORDER BY 2 DESC LIMIT 1
""").fetchone()[0]
n_missing_emp = con.sql("SELECT count(*) FROM windowed WHERE emp_length IS NULL").fetchone()[0]
n_total = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
print(f"emp_length mode (imputation value): {emp_length_mode}")
print(f"rows getting emp_length_was_missing=1: {n_missing_emp:,} ({n_missing_emp/n_total:.1%})")


emp_length mode (imputation value): 10+ years
rows getting emp_length_was_missing=1: 70,579 (5.9%)


**What the output shows:**
```
emp_length mode (imputation value): 10+ years
rows getting emp_length_was_missing=1: 70,579 (5.9%)
```
Matches v2 exactly -- 70,579 rows
(5.9%) will carry `emp_length_was_missing=1`
into the final table, preserving the informative-missingness signal.

**Next:** applying the log transform, restricted to only the 9 fields that
notebook 09 confirmed actually improve target correlation.

## Cell 4 -- log transform, restricted to the 9 validated fields

**What / why:** the pre-v2 cleaning pass applied `log1p` to 16 skewed
fields based on skewness reduction alone. Notebook 09 checked whether the
log version actually has a *stronger* relationship with `is_bad` than the
raw version, and found only 9 of the 16 do. This rebuild applies the
transform only where it's justified -- the other 7 fields keep their raw
form only (no `_log` column at all), rather than carrying an unused,
misleading column into the final table.

**How:** for each of the 9 `LOG_FIELDS`, compute skew before/after as a
sanity check, then apply `log1p` to build the actual `_log` column used
downstream.

**Expect:** skew reduction similar to v2 for these 9 fields -- the transform
behavior itself hasn't changed in v3.

In [5]:
skew_check = con.sql(f"""
    SELECT {', '.join(f"skewness(TRY_CAST({c} AS DOUBLE)) AS {c}_skew" for c in LOG_FIELDS)}
    FROM windowed
""").df()
print("skew before log transform (9 validated fields only):")
print(skew_check.T.rename(columns={0: "skew_before"}).round(2).to_string())
skew_check.T.rename(columns={0: "skew_before"}).to_csv(os.path.join(ASSETS_TABLES, "clean01_skew_check.csv"))
print(f"\nfields getting a _log version: {len(LOG_FIELDS)} (of {len(NUMERIC_KEEP)} total numeric fields)")
TESTED_NOT_LOGGED = ["acc_open_past_24mths", "bc_open_to_buy", "num_actv_rev_tl", "open_acc", "revol_bal", "tot_cur_bal"]
print(f"fields tested but staying raw-only (log did not help, notebook 09): {TESTED_NOT_LOGGED}")
print(f"fields never log candidates in the first place (loan_amnt, int_rate, dti, revol_util): not log-transformed by design, not by test result")


skew before log transform (9 validated fields only):
                           skew_before
annual_inc_skew                  47.23
fico_range_low_skew               1.35
delinq_2yrs_skew                  5.51
inq_last_6mths_skew               1.73
mo_sin_old_rev_tl_op_skew         1.03
mort_acc_skew                     1.63
pub_rec_skew                     11.35
pub_rec_bankruptcies_skew         3.38
total_acc_skew                    0.96

fields getting a _log version: 9 (of 19 total numeric fields)
fields tested but staying raw-only (log did not help, notebook 09): ['acc_open_past_24mths', 'bc_open_to_buy', 'num_actv_rev_tl', 'open_acc', 'revol_bal', 'tot_cur_bal']
fields never log candidates in the first place (loan_amnt, int_rate, dti, revol_util): not log-transformed by design, not by test result


**What the output shows:**
```
skew before log transform (9 validated fields only):
                           skew_before
annual_inc_skew                  47.23
fico_range_low_skew               1.35
delinq_2yrs_skew                  5.51
inq_last_6mths_skew               1.73
mo_sin_old_rev_tl_op_skew         1.03
mort_acc_skew                     1.63
pub_rec_skew                     11.35
pub_rec_bankruptcies_skew         3.38
total_acc_skew                    0.96

fields getting a _log version: 9 (of 19 total numeric fields)
fields tested but staying raw-only (log did not help, notebook 09): ['acc_open_past_24mths', 'bc_open_to_buy', 'num_actv_rev_tl', 'open_acc', 'revol_bal', 'tot_cur_bal']
fields never log candidates in the first place (loan_amnt, int_rate, dti, revol_util): not log-transformed by design, not by test result
```
9 fields will get a `_log` column in the final table --
down from 16 in the pre-v2 pass. The dropped fields (`acc_open_past_24mths`,
`bc_open_to_buy`, `num_actv_rev_tl`, `open_acc`, `revol_bal`, `tot_cur_bal`)
keep only their raw form, since notebook 09 found the log version didn't
actually strengthen their relationship with `is_bad`. All 6 of these
raw-only fields are exactly the fields cell 6 winsorizes (along with
`loan_amnt` and `dti`, which were never log candidates in the first place):
a field that gets no variance-taming from a log transform is also one where
a handful of extreme values can dominate a linear model's fit, which is
precisely why notebook 02 cell 8 flagged them for capping instead.

**Next:** engineering the two new features -- `issue_year` (population-drift
signal from notebook 10) and `addr_state_grouped` (small-state handling from
notebook 11) -- then applying the winsorizing decision in cell 6.

## Cell 5 -- new features: issue_year and addr_state_grouped

**What / why:** two direct responses to EDA findings. `issue_year`:
notebook 10 found `int_rate`'s distribution drifted moderately (PSI 0.140)
across the 2013-2017 window, so origination year carries real signal a model
shouldn't be blind to. `addr_state_grouped`: notebook 11 found the smallest
states have single-digit loan counts -- keeping them as their own category
in a categorical encoding would let a model fit noise from a handful of
rows, so states with fewer than 1,000 windowed loans get bucketed into
`OTHER`.

**How:** extract `issue_year` directly from `issue_d`; compute per-state
counts, bucket any state under the 1,000-loan threshold into `OTHER`.

**Expect:** `issue_year` ranging 2013-2017; `addr_state_grouped` with most
states unchanged and only the smallest tail collapsed.

In [6]:
state_counts = con.sql("SELECT addr_state, count(*) n FROM windowed GROUP BY 1").df()
small_states = state_counts[state_counts["n"] < 1000]["addr_state"].tolist()
print(f"states below the 1,000-loan threshold, bucketed to OTHER: {small_states}")
print(f"({len(small_states)} of {len(state_counts)} states)")

year_check = con.sql("SELECT DISTINCT substr(issue_d,-4) AS issue_year FROM windowed ORDER BY 1").df()
print(f"\nissue_year values: {year_check['issue_year'].tolist()}")


states below the 1,000-loan threshold, bucketed to OTHER: ['IA']
(1 of 51 states)

issue_year values: ['2013', '2014', '2015', '2016', '2017']


**What the output shows:**
```
states below the 1,000-loan threshold, bucketed to OTHER: ['IA']
(1 of 51 states)

issue_year values: ['2013', '2014', '2015', '2016', '2017']
```
1 state(s) get bucketed into `OTHER` for
`addr_state_grouped` -- a small tail,
consistent with notebook 11's finding that only the smallest states lack
enough volume. `issue_year` confirms the expected 2013-2017 range, ready to
use as either a categorical or ordinal feature in Phase 1.

**Next:** applying notebook 02 cell 8's outlier-treatment decision for
real -- winsorizing the 8 flagged fields at the 1st/99th percentile.

## Cell 6 -- winsorize the 8 flagged fields (gap-closure: outlier treatment)

**What / why:** notebook 02 cells 3/4 counted outliers per numeric feature
but v2 never turned that into a treatment decision -- notebook 02's
gap-closure cell 8 made that decision (cap at the 1st/99th percentile for 8
unbounded, untransformed dollar/count fields where a log transform didn't
help), but a decision documented in an EDA notebook doesn't change the data
until it's actually applied here, the one notebook allowed to do that.

**How:** compute the 1st and 99th percentile of each `CAP_FIELDS` column on
`windowed` (via DuckDB's `percentile_cont`), then report how many rows would
be clamped at each bound -- so the decision's actual impact is visible before
it's baked into the final table in cell 7.

**Expect:** a small, single-digit percentage of rows affected per field (by
construction, capping at the 1st/99th percentile clamps roughly 2% of rows
combined, less if a field's distribution is already tight near those
bounds) -- this reduces the leverage of extreme values without discarding
any rows.

In [7]:
cap_bounds = {}
cap_impact_rows = []
for col in CAP_FIELDS:
    p01, p99 = con.sql(f"""
        SELECT
            quantile_cont(TRY_CAST({col} AS DOUBLE), 0.01),
            quantile_cont(TRY_CAST({col} AS DOUBLE), 0.99)
        FROM windowed
    """).fetchone()
    cap_bounds[col] = (p01, p99)
    n_below, n_above, n_total_col = con.sql(f"""
        SELECT
            sum(CASE WHEN TRY_CAST({col} AS DOUBLE) < {p01} THEN 1 ELSE 0 END),
            sum(CASE WHEN TRY_CAST({col} AS DOUBLE) > {p99} THEN 1 ELSE 0 END),
            count(*)
        FROM windowed
    """).fetchone()
    cap_impact_rows.append({
        "field": col, "p01": round(p01, 2), "p99": round(p99, 2),
        "n_capped_low": n_below, "n_capped_high": n_above,
        "pct_capped": round(100 * (n_below + n_above) / n_total_col, 3),
    })

cap_impact = pd.DataFrame(cap_impact_rows)
print(cap_impact.to_string(index=False))
cap_impact.to_csv(os.path.join(ASSETS_TABLES, "clean01_cap_impact.csv"), index=False)
print(f"\ntotal fields winsorized: {len(CAP_FIELDS)}")
print(f"average share of rows capped per field: {cap_impact['pct_capped'].mean():.2f}%")


               field     p01       p99  n_capped_low  n_capped_high  pct_capped
           loan_amnt 1600.00  35000.00         11125           6800       1.499
                 dti    2.06     38.40         11896          11903       1.990
            open_acc    3.00     29.00          3874          11414       1.278
           revol_bal  270.00  96555.30         11953          11959       2.000
         tot_cur_bal 3277.00 672525.40         11952          11959       1.999
      bc_open_to_buy    0.00  72836.05             0          11835       0.990
acc_open_past_24mths    0.00     15.00             0           9204       0.770
     num_actv_rev_tl    1.00     17.00          4456           9093       1.133

total fields winsorized: 8
average share of rows capped per field: 1.46%


**What the output shows:**
```
field     p01       p99  n_capped_low  n_capped_high  pct_capped
           loan_amnt 1600.00  35000.00         11125           6800       1.499
                 dti    2.06     38.40         11896          11903       1.990
            open_acc    3.00     29.00          3874          11414       1.278
           revol_bal  270.00  96555.30         11953          11959       2.000
         tot_cur_bal 3277.00 672525.40         11952          11959       1.999
      bc_open_to_buy    0.00  72836.05             0          11835       0.990
acc_open_past_24mths    0.00     15.00             0           9204       0.770
     num_actv_rev_tl    1.00     17.00          4456           9093       1.133

total fields winsorized: 8
average share of rows capped per field: 1.46%
```
Each field clamps close to the expected ~2% combined (1% per tail) --
confirming the bounds are doing what a 1st/99th percentile cap is supposed
to do: trim the genuine extremes without touching the bulk of the
distribution. These bounds (`cap_bounds`) are applied for real in cell 7,
replacing each field's raw value with `LEAST(GREATEST(value, p01), p99)`.

**Next:** assembling every piece -- imputed numerics, the restricted log
set, the emp_length flag, the two new engineered features, and now the
winsorized values -- into the final table and writing it to
`data/03_processed/`.

## Cell 7 -- assemble the final table

**What / why:** this is where every decision from cells 1-6 gets applied at
once, in SQL, against the full windowed population -- pulling it together
here rather than in pandas keeps the transformation logic auditable as a
single query.

**How:** build a `SELECT` that applies median imputation (via `COALESCE`),
winsorizing (via `LEAST`/`GREATEST` with the cell 6 bounds -- applied after
imputation, so an imputed median value is itself subject to the same clamp
as a real one, though medians are never near the 1st/99th percentile tails
in practice), the emp_length mode-fill and flag, the restricted log
transform, and the two new engineered columns; drops `avg_cur_bal` and
`emp_title` (never retained) by simply not selecting them. Categorical
columns are selected as-is -- no encoding is applied (see cell 8). Writes
the result to `data/03_processed/lendingclub_model_ready.parquet`.

**Expect:** a table with the same row count as `windowed`, and a column
count reflecting: 19 numeric (down from 20, 8 of them now winsorized) + 9
log versions + 7 categorical + `emp_length_was_missing` + `issue_year` +
`addr_state_grouped` + `id`/`issue_d`/`loan_status`/`is_bad` -- unchanged
from v2's 42, since winsorizing changes values, not the column count.

In [8]:
other_case = "('" + "','".join(small_states) + "')" if small_states else "('__none__')"

numeric_select = []
for col in NUMERIC_KEEP:
    if col in medians:
        base = f"COALESCE(TRY_CAST({col} AS DOUBLE), {medians[col]})"
    else:
        base = f"TRY_CAST({col} AS DOUBLE)"
    if col in cap_bounds:
        p01, p99 = cap_bounds[col]
        base = f"LEAST(GREATEST({base}, {p01}), {p99})"
    numeric_select.append(f"{base} AS {col}")
    if col in LOG_FIELDS:
        # log transform is computed from the same (imputed, and winsorized if
        # applicable) base as the raw column, so the two stay consistent
        numeric_select.append(f"ln(1 + GREATEST({base}, 0)) AS {col}_log")

select_clause = ",\n    ".join(
    ["id", "issue_d", "loan_status", "is_bad"] +
    numeric_select +
    ["term", "grade",
     f"COALESCE(emp_length, '{emp_length_mode}') AS emp_length",
     "CASE WHEN emp_length IS NULL THEN 1 ELSE 0 END AS emp_length_was_missing",
     "home_ownership", "verification_status", "purpose",
     "addr_state",
     f"CASE WHEN addr_state IN {other_case} THEN 'OTHER' ELSE addr_state END AS addr_state_grouped",
     "CAST(substr(issue_d, -4) AS INT) AS issue_year"]
)

final_df = con.sql(f"SELECT\n    {select_clause}\nFROM windowed").df()
print(f"final shape: {final_df.shape[0]:,} rows x {final_df.shape[1]} columns")
print(f"\ncolumns: {final_df.columns.tolist()}")

import os
os.makedirs("../../data/03_processed", exist_ok=True)
final_df.to_parquet("../../data/03_processed/lendingclub_model_ready.parquet", index=False)
print(f"\nwrote {final_df.shape[0]:,} rows to data/03_processed/lendingclub_model_ready.parquet")


final shape: 1,195,879 rows x 42 columns

columns: ['id', 'issue_d', 'loan_status', 'is_bad', 'loan_amnt', 'int_rate', 'annual_inc', 'annual_inc_log', 'dti', 'fico_range_low', 'fico_range_low_log', 'delinq_2yrs', 'delinq_2yrs_log', 'inq_last_6mths', 'inq_last_6mths_log', 'open_acc', 'pub_rec', 'pub_rec_log', 'revol_bal', 'revol_util', 'total_acc', 'total_acc_log', 'mort_acc', 'mort_acc_log', 'pub_rec_bankruptcies', 'pub_rec_bankruptcies_log', 'tot_cur_bal', 'bc_open_to_buy', 'acc_open_past_24mths', 'mo_sin_old_rev_tl_op', 'mo_sin_old_rev_tl_op_log', 'num_actv_rev_tl', 'term', 'grade', 'emp_length', 'emp_length_was_missing', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'addr_state_grouped', 'issue_year']

wrote 1,195,879 rows to data/03_processed/lendingclub_model_ready.parquet


**What the output shows:**
```
final shape: 1,195,879 rows x 42 columns

columns: ['id', 'issue_d', 'loan_status', 'is_bad', 'loan_amnt', 'int_rate', 'annual_inc', 'annual_inc_log', 'dti', 'fico_range_low', 'fico_range_low_log', 'delinq_2yrs', 'delinq_2yrs_log', 'inq_last_6mths', 'inq_last_6mths_log', 'open_acc', 'pub_rec', 'pub_rec_log', 'revol_bal', 'revol_util', 'total_acc', 'total_acc_log', 'mort_acc', 'mort_acc_log', 'pub_rec_bankruptcies', 'pub_rec_bankruptcies_log', 'tot_cur_bal', 'bc_open_to_buy', 'acc_open_past_24mths', 'mo_sin_old_rev_tl_op', 'mo_sin_old_rev_tl_op_log', 'num_actv_rev_tl', 'term', 'grade', 'emp_length', 'emp_length_was_missing', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'addr_state_grouped', 'issue_year']

wrote 1,195,879 rows to data/03_processed/lendingclub_model_ready.parquet
```
1,195,879 rows x 42 columns -- same
shape as v2 (winsorizing changes values within existing columns, not the
column count), with the 8 `CAP_FIELDS` now clamped to their 1st/99th
percentile bounds from cell 6, and every other column traceable to a
specific decision in cells 1-5.

**Next:** validating the output -- the same checks v2 ran, plus a
duplicate-id check on this notebook's own final table (gap-closure) and a
confirmation that the winsorizing was actually applied.

## Cell 8 -- validate

**What / why:** the same validation v2 ran, plus two new checks specific to
this rebuild's gap-closure work -- `id` uniqueness on the final table itself
(notebook 02 cell 6 checked `windowed`, but never re-checked after this
notebook's own transformations could in principle have introduced a
duplicate via a join or group-by bug) and confirmation that winsorizing
actually took effect (each capped field's max should now equal its cell 6
`p99` bound, not the original unbounded max).

**How:** count total missing values, compare row count to `windowed`, check
bad rate, check the two v2 structural changes, check `id` uniqueness, and
check that each `CAP_FIELDS` column's max matches its winsorizing bound.

**Expect:** zero missing values, matching row count, unchanged bad rate
(~20.5%), `avg_cur_bal` absent, `issue_year`/`addr_state_grouped` present,
`id` unique, and every capped field's max at (or below, from floating-point
rounding) its `p99` bound.

In [9]:
total_missing = final_df.isna().sum().sum()
row_match = len(final_df) == n_rows
bad_rate_final = final_df["is_bad"].mean()
avg_cur_bal_absent = "avg_cur_bal" not in final_df.columns and "avg_cur_bal_log" not in final_df.columns
new_cols_present = "issue_year" in final_df.columns and "addr_state_grouped" in final_df.columns
id_unique = final_df["id"].nunique() == len(final_df)
capping_applied = all(final_df[col].max() <= cap_bounds[col][1] + 1e-6 for col in CAP_FIELDS)

print(f"total missing values in final table: {total_missing}")
print(f"row count matches windowed: {row_match}")
print(f"bad rate in final table: {bad_rate_final:.1%}")
print(f"avg_cur_bal correctly absent: {avg_cur_bal_absent}")
print(f"new engineered columns present: {new_cols_present}")
print(f"log-transformed column count: {sum(c.endswith('_log') for c in final_df.columns)} (expect {len(LOG_FIELDS)})")
print(f"id unique on final table: {id_unique} ({final_df['id'].nunique():,} distinct of {len(final_df):,} rows)")
print(f"winsorizing correctly applied to all {len(CAP_FIELDS)} capped fields: {capping_applied}")

all_checks_passed = (total_missing == 0 and row_match and avg_cur_bal_absent and new_cols_present
                      and sum(c.endswith('_log') for c in final_df.columns) == len(LOG_FIELDS)
                      and id_unique and capping_applied)
print(f"\nall checks passed: {all_checks_passed}")

print()
print("Categorical encoding: intentionally NOT applied in this table.")
print(f"Unencoded categorical columns carried through as-is: {CATEGORICAL_KEEP + ['addr_state_grouped']}")
print("Encoding (one-hot / ordinal / target) is a Phase 1 modeling decision -- see")
print("notebooks/02_eda/14_eda_governance_synthesis_reporting.ipynb cell 7 for the")
print("per-field cardinality and recommended-encoding table this defers to.")


total missing values in final table: 0
row count matches windowed: True
bad rate in final table: 20.5%
avg_cur_bal correctly absent: True
new engineered columns present: True
log-transformed column count: 9 (expect 9)
id unique on final table: True (1,195,879 distinct of 1,195,879 rows)
winsorizing correctly applied to all 8 capped fields: True

all checks passed: True

Categorical encoding: intentionally NOT applied in this table.
Unencoded categorical columns carried through as-is: ['term', 'grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'addr_state_grouped']
Encoding (one-hot / ordinal / target) is a Phase 1 modeling decision -- see
notebooks/02_eda/14_eda_governance_synthesis_reporting.ipynb cell 7 for the
per-field cardinality and recommended-encoding table this defers to.


**What the output shows:**
```
total missing values in final table: 0
row count matches windowed: True
bad rate in final table: 20.5%
avg_cur_bal correctly absent: True
new engineered columns present: True
log-transformed column count: 9 (expect 9)
id unique on final table: True (1,195,879 distinct of 1,195,879 rows)
winsorizing correctly applied to all 8 capped fields: True

all checks passed: True

Categorical encoding: intentionally NOT applied in this table.
Unencoded categorical columns carried through as-is: ['term', 'grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'addr_state_grouped']
Encoding (one-hot / ordinal / target) is a Phase 1 modeling decision -- see
notebooks/02_eda/14_eda_governance_synthesis_reporting.ipynb cell 7 for the
per-field cardinality and recommended-encoding table this defers to.
```
All checks passed -- the rebuilt data/03_processed/lendingclub_model_ready.parquet is validated and ready for Phase 1, now incorporating every actionable finding from the full 14-category EDA suite, including all four gaps the depth/sufficiency review found.
Row count and bad rate still match `windowed` exactly (winsorizing changes
extreme values, not which rows exist or how many are bad), `id` is
confirmed unique on the notebook's own final output (not just on
`windowed` upstream), and every capped field's maximum now sits at its
1st/99th-percentile bound. The categorical fields are confirmed still
present and unencoded, by design -- the encoding decision itself, with
reasoning per field, lives in notebook 14 cell 7, not duplicated here.

**Next:** this closes all four gaps the depth/sufficiency review found --
duplicates and leakage re-validation (notebook 02, cells 6-7), outlier
treatment (notebook 02 cell 8, applied here in cells 6-7), and categorical
encoding strategy (notebook 14 cell 7). Lending Club's Phase 0 is complete:
ingestion, all 14 EDA categories at appropriate depth with no known gaps,
and a cleaning pass grounded in the full EDA and validated end to end.